| <div style="width:330px"><img src="https://www.ufz.de/static/custom/weblayout/DefaultInternetLayout/img/logos/ufz_transparent_de_blue.png" width="300" alt="UFZ"/></div> | <div style="width:290px"><img src="https://discourse.opengeosys.org/uploads/default/original/1X/a288c27cc8f73e6830ad98b8729637a260ce3490.png" width="290" alt="OpenGeoSys"/></div> | <div style="width:330px"><img src="https://github.com/nagelt/Teaching_Scripts/raw/9d9e29ecca4b04eaf7397938eacbf116d37ddc93/Images/TUBAF_Logo_blau.png" width="300" alt="TU Bergakademie Freiberg"/></div> |
|---|---|--:|


# Geostatic Initialisation of Layered Ground Using the Modified Cam-Clay Model
**Dr. Pavan Kumar Bhukya and Prof. Dr. Thomas Nagel**  
Chair of Soil Mechanics and Foundation Engineering · TU Bergakademie Freiberg

## Description

This OpenGeoSys benchmark verifies a prescribed $K_0$ geostatic initialisation of horizontally layered ground using the Modified Cam-Clay (MCC) model implemented through MFront [1]. The present-day vertical effective stress is obtained from self-weight and surface surcharge, while the horizontal stress is prescribed from an at-rest coefficient $K_0$ linked to the assumed overconsolidation history. Compatible preconsolidation pressure and specific volume are then assigned at integration points.

Three vertical overconsolidation ratios are compared under identical present-day loading. The resulting $K_0$ states are held under gravity and surface loading with hardening enabled. Analytical stress profiles, bottom reactions, displacement, yield admissibility, and material-state preservation provide complementary checks that the prescribed state remains in equilibrium without unintended constitutive evolution.

The benchmark therefore separates geostatic state construction from subsequent engineering loading. The verified state provides a consistent starting point for subsequent geotechnical analyses. Existing ground can be loaded or excavated from this state, while newly constructed systems on the layered ground, such as embankments, backfills, or retaining structures, require an appropriate placement or activation state before their self-weight and construction loads are equilibrated.

## Install the notebook environment

This is a setup-enabled copy of the MCC geostatic teaching notebook. The original
notebook is unchanged. Use Linux x86_64 and Python 3.12 for the private HM toolkit
prototype; place its supplied wheel beside this notebook or set OGS_HM_WHEEL.
Upstream OGSTools and OGS run this SMALL_DEFORMATION example and its mesh utility.
The custom HM solver is also installed, ready for later HYDRO_MECHANICS examples
via `hm.run(...)`. It cannot replace the solver in this MCC example because its
build enables only HydroMechanics. No development repository or result files
are needed to run the notebook. Internet access is needed for Python packages.

Run the installation cell in a fresh kernel. If packages were already imported,
restart the kernel after installation and continue below. The local wheel path
will become a pinned download URL when the toolkit is released.

In [ ]:
import os, platform, sys, subprocess
from pathlib import Path
assert platform.system() == "Linux" and platform.machine() == "x86_64", "Use Linux x86_64 for this prototype."
assert sys.version_info[:2] == (3, 12), "Select a Python 3.12 kernel."
configured = os.environ.get("OGS_HM_WHEEL")
if configured:
    wheel = Path(configured).expanduser().resolve()
else:
    candidates = sorted({p.resolve() for folder in [Path.cwd(), Path.cwd()/"dist", Path.cwd().parent/"dist"]
                         for p in folder.glob("ogs_hm_tools-0.1.0.dev0-cp312-cp312-linux_x86_64.whl")})
    if len(candidates) != 1:
        raise FileNotFoundError("Place the supplied wheel beside the notebook, or set OGS_HM_WHEEL.")
    wheel = candidates[0]
assert wheel.is_file(), wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", str(wheel), "gmsh==4.15.2", "pandas==3.0.5", "matplotlib==3.10.9", "pyvista==0.49.0", "ipykernel==7.3.0"])
# Keep upstream OGS commands in this kernel environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")


In [ ]:
import ogs_hm_tools as hm
hm.doctor()

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import gmsh
import matplotlib.pyplot as plt
import numpy as np
import ogstools as ot
import pandas as pd
import pyvista as pv
from IPython.display import display
from matplotlib.lines import Line2D

out_dir = Path(os.environ.get("OGS_TESTRUNNER_OUT_DIR", "_out"))
out_dir.mkdir(parents=True, exist_ok=True)

work = out_dir / "runs"
work.mkdir(exist_ok=True)

integration_order = 4
cli = ot.cli()

# Plot styling
plt.rcParams.update(
    {
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "font.size": 9,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

## 1. Ground model and loading

A horizontally layered ground profile is analysed under plane-strain conditions using the `SmallDeformation` process in OpenGeoSys. The layer thicknesses and unit weights are shown below. Dry conditions are assumed, such that pore pressure is zero and total and effective stresses are identical. Geometry, unit weights, and external loading are kept unchanged across all OCR cases (OCR = 1, 2, 5); only the prescribed stress history and the associated MCC state variables differ.

A uniform sustained pressure is applied at the ground surface and included consistently in both the analytical stress profiles and the numerical boundary conditions. It also maintains a positive mean effective stress near the surface, avoiding the zero-pressure limit of standard Modified Cam-Clay [1,2]. The surface pressure is therefore treated as part of the physical problem definition rather than as a numerical regularisation.

In [ ]:
layers = pd.DataFrame(
    [
        {"Layer": "Upper soil", "Thickness [m]": 2.0, "Unit weight [kN/m³]": 18.0},
        {"Layer": "Middle soil", "Thickness [m]": 2.0, "Unit weight [kN/m³]": 17.0},
        {"Layer": "Lower soil", "Thickness [m]": 4.0, "Unit weight [kN/m³]": 20.0},
        # Add another row here for another layer.
    ]
)

width = 12.0          # m
surface_pressure = 10.0  # kPa; held constant
mesh_size = 0.5       # m; suitable for this equilibrium demonstration

assert len(layers) > 0 and np.all(layers.iloc[:, 1:].to_numpy(dtype=float) > 0)
assert width > 0 and surface_pressure > 0 and mesh_size > 0

thickness = layers["Thickness [m]"].to_numpy()
gamma = layers["Unit weight [kN/m³]"].to_numpy() * 1000
edges = np.r_[0, np.cumsum(thickness)]
depth = edges[-1]

display(layers)

# A horizontal bar for each layer gives its true thickness and depth.
fig, (ax, aw) = plt.subplots(1, 2, figsize=(10, 4.8), constrained_layout=True)

centres = (edges[:-1] + edges[1:]) / 2

ax.barh(
    centres,
    width,
    height=thickness,
    color=plt.colormaps["Pastel1"](np.linspace(0, 0.8, len(layers))),
    edgecolor="0.4",
)

for i, row in layers.iterrows():
    ax.text(
        width / 2,
        centres[i],
        f"{row['Layer']}\n{row['Unit weight [kN/m³]']:g} kN/m³",
        ha="center",
        va="center",
    )

# Ground surface
ax.hlines(0, 0, width, color="0.2", linewidth=1.5)

# Draw downward arrows to indicate uniform surcharge
n_arrows = 7
x_arrows = np.linspace(0.8, width - 0.8, n_arrows)

for x in x_arrows:
    ax.annotate(
        "",
        xy=(x, 0.02),
        xytext=(x, -0.65),
        arrowprops=dict(arrowstyle="-|>", lw=1.2, color="tab:red"),
        annotation_clip=False,
    )

ax.text(
    width / 2,
    -0.82,
    rf"Uniform surcharge, $q = {surface_pressure:g}\ \mathrm{{kPa}}$",
    ha="center",
    va="center",
)

aw.stairs(
    gamma / 1000,
    edges,
    orientation="horizontal",
    baseline=None,
    color="tab:blue",
    linewidth=2,
)

ax.set(
    xlim=(0, width),
    ylim=(depth, -1.1),   # extend above the ground surface to show arrows
    xlabel="Width [m]",
    ylabel="Depth [m]",
)

aw.set(
    ylim=(depth, 0),
    xlabel="Unit weight [kN/m³]",
    ylabel="Depth [m]",
)

# Remove the box around the surcharge region
ax.spines["top"].set_visible(False)
ax.spines["left"].set_bounds(0, depth)
ax.spines["right"].set_visible(False)

plt.show()

### Constitutive properties and stress history

All layers use the same Modified Cam-Clay parameters; only the unit weight varies with depth. The values used here are assumed verification inputs rather than a calibration to a specific soil. The main MCC parameters are:

- $M$: critical-state stress ratio,
- $\lambda$: slope of the virgin compression line,
- $\kappa$: slope of the unloading/reloading line,
- $\xi$: elastic Poisson's ratio,
- $N$: specific volume on the isotropic normal compression line at the reference pressure $p_{\mathrm{ref}} = 1\,\mathrm{kPa}$.

The constitutive response is evaluated with the MFront/OpenGeoSys `ModCamClay_semiExpl` implementation [1]. The stress history is described through the vertical overconsolidation ratio,

$$
\mathrm{OCR}
=
\frac{\sigma'_{v,\max}}{\sigma'_{v,0}},
$$

where $\sigma'_{v,\max}$ is the maximum past vertical effective stress and $\sigma'_{v,0}$ is the current vertical effective stress. Three cases are considered: $\mathrm{OCR}=1$, $2$, and $5$.

For each OCR, the normally consolidated at-rest coefficient is first estimated from the critical-state parameter $M$. The conversion between $M$ and $\phi'$ uses the triaxial-compression convention [2]:

$$
K_{0,\mathrm{NC}} = 1-\sin\phi',
\qquad
\sin\phi' = \frac{3M}{6+M}.
$$

The normally consolidated $K_0$ is prescribed for this benchmark rather than obtained by simulating one-dimensional compression with MCC. Assuming that the maximum past vertical stress had the lateral stress ratio $K_{0,\mathrm{NC}}$, the initial lateral stress coefficient follows the automatic stress prescription for advanced constitutive models [3]:

$$
K_0
=
K_{0,\mathrm{NC}}\,\mathrm{OCR}
-
\frac{\xi}{1-\xi}
\left(\mathrm{OCR}-1\right),
$$

with

$$
\sigma'_{h,0}
=
K_0\,\sigma'_{v,0}.
$$

This expression corresponds to one-dimensional elastic unloading under zero lateral strain,

$$
d\sigma'_h
=
\frac{\xi}{1-\xi}\,d\sigma'_v.
$$

Here, $\xi$ denotes the elastic Poisson's ratio; a nonstandard symbol is used to avoid confusion with the MCC specific volume $v$.

Integrating this increment from $\sigma'_{v,\max}=\mathrm{OCR}\,\sigma'_{v,0}$ to the current vertical stress gives the expression for $K_0$ above. For $\mathrm{OCR}=1$, the relation reduces to $K_0=K_{0,\mathrm{NC}}$. For overconsolidated states, the increase in $K_0$ reflects the assumed unloading history.

This relation is used here as a prescribed unloading estimate rather than as a universal correlation for natural soils. A measured or independently specified $K_0$ may be used instead, provided that the resulting MCC state satisfies the admissibility checks introduced below.

In [ ]:
M = 1.2      # Critical-state slope
lam = 0.15   # Virgin compression index
kappa = 0.03 # Swelling index
xi = 0.3     # Poisson's ratio
N = 3.2      # Reference specific volume

OCR_values = [1, 2, 5]  # Compare these histories on the SAME ground and mesh.
reference_pressure = 1_000.0  # Pa
K0_input = None  # None: automatic OCR-dependent K0; otherwise a known value

assert 0 < M < 3 and 0 < kappa < lam and 0 <= xi < 0.5
assert all(np.isfinite(value) and value >= 1 for value in OCR_values)
K0_nc = 1 - 3 * M / (6 + M)

def lateral_ratio(ocr):
    # Elastic-unloading estimate; a measured K0 can override it.
    return (
        K0_nc * ocr - xi / (1 - xi) * (ocr - 1) if K0_input is None else float(K0_input)
    )


assert all(np.isfinite(lateral_ratio(o)) and lateral_ratio(o) > 0 for o in OCR_values)
display(pd.DataFrame({"OCR": OCR_values, "K0": [lateral_ratio(o) for o in OCR_values]}))

## 2. Compatible initial state

The prescribed stress field must be accompanied by compatible Modified Cam-Clay state variables. In this benchmark, the preconsolidation pressure and specific volume are constructed from the assumed stress history before the equilibrium hold. Taking depth $z$ as positive downward and compression as positive,

$$
\sigma'_{v,0}(z)
=
q_s+\int_0^z \gamma(\zeta)\,d\zeta,
\qquad
\sigma'_{h,0}(z)
=
K_0\,\sigma'_{v,0}(z),
$$

where $q_s$ is the sustained surface surcharge and $\gamma$ is the dry unit weight. The corresponding stress invariants are

$$
p_0
=
\frac{\sigma'_{v,0}+2\sigma'_{h,0}}{3},
\qquad
q_0
=
\left|\sigma'_{v,0}-\sigma'_{h,0}\right|.
$$

Here, $q_0$ denotes deviatoric stress, while $q_s$ denotes the applied surface surcharge. The two horizontal normal stresses are assumed equal.

### Preconsolidation pressure

The maximum past vertical effective stress is prescribed from the vertical OCR:

$$
\sigma'_{v,\max}
=
\mathrm{OCR}\,\sigma'_{v,0}.
$$

The lateral stress ratio is taken as $K_{0,\mathrm{NC}}$, giving

$$
p_{\max}
=
\frac{1+2K_{0,\mathrm{NC}}}{3}\,\sigma'_{v,\max},
\qquad
q_{\max}
=
\left|1-K_{0,\mathrm{NC}}\right|\sigma'_{v,\max}.
$$

The assumed historical state is placed on the MCC yield surface [1,2],

$$
f
=
q^2+M^2p(p-p_c)
=
0,
$$

which gives the initial cap pressure

$$
p_{c0}
=
p_{\max}
+
\frac{q_{\max}^2}{M^2p_{\max}}
+
\delta p_c,
\qquad
\delta p_c=1\,\mathrm{Pa}.
$$

The small offset $\delta p_c$ places the reference state marginally inside the yield surface and avoids numerical ambiguity at exact yield. It is not an applied load. This construction assumes elastic unloading that preserves the cap pressure. However, it is important to note that the vertical OCR and the cap ratio $p_{c0}/p_0$ are distinct [1].

### Compatible specific volume

The initial specific volume follows the MCC compression and unloading relations [1,2]:

$$
v_0
=
N
-
\lambda
\ln\left(
\frac{p_{c0}}{p_{\mathrm{ref}}}
\right)
+
\kappa
\ln\left(
\frac{p_{c0}}{p_0}
\right),
\qquad
p_{\mathrm{ref}}
=
1\,\mathrm{kPa}.
$$

Since

$$
v_0=1+e_0,
$$

the notebook checks that $v_0>1$ and that the current stress state lies inside or on the assigned yield surface. The resulting $\{\boldsymbol{\sigma}'_0,p_{c0},v_0\}$ set defines the complete initial MCC state used by OGS. The subsequent equilibrium hold then tests whether this prescribed state remains unchanged under the applied loads.

In [ ]:
def expected(z, ocr):
    # Integrate each layer's weight, then compute the current and historical states.
    sv = np.full_like(np.asarray(z, dtype=float), surface_pressure * 1000)
    for top, h, g in zip(edges[:-1], thickness, gamma):
        sv += g * np.clip(np.asarray(z) - top, 0, h)
    sh = lateral_ratio(ocr) * sv
    p, q = (sv + 2 * sh) / 3, abs(sv - sh)
    p_max, q_max = ocr * sv * (1 + 2 * K0_nc) / 3, ocr * sv * abs(1 - K0_nc)
    pc = p_max + q_max * q_max / (M * M * p_max) + 1.0
    v = N - lam * np.log(pc / reference_pressure) + kappa * np.log(pc / p)
    assert np.all(v > 1) and np.all(
        (q * q + M * M * p * (p - pc)) / pc**2 <= 1e-10
    ), "Incompatible MCC inputs."
    return sv, sh, pc, v


# This checks all three proposed states before any simulation is started.
for ocr in OCR_values:
    expected(np.linspace(0, depth, 101), ocr)


## 3. Numerical model

A conforming mesh of six-node quadratic triangular elements (T6) aligns with the layer interfaces. Horizontal displacement is restrained along the lateral boundaries, and both displacement components are fixed at the bottom. The surface carries the uniform pressure $q_s$. The prescribed initial fields are evaluated at integration points using the `ModCamClay_semiExpl` behaviour within the `SmallDeformation` process. Gravity and surface pressure are fully active from the start, and hardening remains enabled. A one second pseudo time hold under constant loading tests whether the assigned state remains in equilibrium. The loads remain constant throughout this static consistency check. The following cells generate the mesh and project files, run the hold and compare the numerical results with the analytical reference.

In [ ]:
# Create one mesh shared by all OCR cases; fragment joins the layer interfaces.
gmsh.initialize()
try:
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("ground")
    occ = gmsh.model.occ
    regions = [
        (2, occ.addRectangle(0, -edges[i + 1], 0, width, h))
        for i, h in enumerate(thickness)
    ]
    if len(regions) > 1:
        occ.fragment([regions[0]], regions[1:])
    occ.synchronize()
    gmsh.model.addPhysicalGroup(
        2, [tag for dim, tag in gmsh.model.getEntities(2)], name="soil"
    )
    # Label the four external boundaries for the OGS boundary conditions.
    boxes = {
        tag: gmsh.model.getBoundingBox(dim, tag)
        for dim, tag in gmsh.model.getEntities(1)
    }
    for name, axis, location in [
        ("left", 0, 0),
        ("right", 0, width),
        ("bottom", 1, -depth),
        ("top", 1, 0),
    ]:
        # A boundary lies entirely on its prescribed x or y coordinate.
        tags = [
            tag
            for tag, box in boxes.items()
            if np.allclose([box[axis], box[axis + 3]], location, rtol=0, atol=1e-6)
        ]
        gmsh.model.addPhysicalGroup(1, tags, name=name)
    gmsh.model.mesh.setSize(gmsh.model.getEntities(0), mesh_size)
    gmsh.option.setNumber("Mesh.ElementOrder", 2)
    gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
    gmsh.model.mesh.generate(2)
    gmsh.write(str(work / "ground.msh"))
finally:
    gmsh.finalize()
# OGSTools converts Gmsh groups and supplies the OGS boundary-node mappings.
meshes = ot.Meshes.from_gmsh(work / "ground.msh", meshname="soil", log=False)
for name, mesh in meshes.items():
    name = name.removeprefix("physical_group_")
    mesh.save(work / f"{name}.vtu")
soil = meshes["soil"]
print(f"{soil.n_cells} T6 elements; {len(layers)} layers. Files: {work}")


### 3.1. Mesh and observation locations

The shaded central strip identifies the raw integration points used in depth-profile figures. Its centre is $x=W/2$ and its half-width is half the chosen mesh size. Profile markers retain their actual depths; no interpolation onto the dashed centreline is performed. Verification evaluates the whole domain. The later yield diagnostic selects its point independently, using the highest mean effective stress.

In [ ]:
# Plot the generated triangle corners, not a schematic mesh.
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
triangles = soil.cells.reshape(-1, 7)[:, 1:4]
ax.triplot(
    soil.points[:, 0], -soil.points[:, 1], triangles, color="0.65", linewidth=0.4
)
for boundary in edges[1:-1]:
    ax.axhline(boundary, color="tab:blue", linewidth=1.3)
ax.axvspan(
    width / 2 - mesh_size / 2,
    width / 2 + mesh_size / 2,
    color="tab:red",
    alpha=0.15,
    label="Material-point sampling strip",
)
ax.axvline(width / 2, color="tab:red", linestyle="--", label=f"x = {width/2:g} m")
ax.set(
    xlim=(0, width),
    ylim=(depth, 0),
    xlabel="Horizontal position [m]",
    ylabel="Depth [m]",
    title=f"{soil.n_cells} T6 elements · same mesh for every OCR",
)
ax.set_aspect("equal")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.show()


### 3.2. Project setup

The common project defines the MCC material, boundary conditions, and a constant-load equilibrium hold using the OGSTools API.

In [ ]:
prj = ot.Project(output_file=out_dir / "ground_template.prj", verbose=False)
for name in ["soil", "left", "right", "bottom", "top"]:
    prj.mesh.add_mesh(filename=f"{name}.vtu")
prj.processes.set_process(
    name="SD",
    type="SMALL_DEFORMATION",
    integration_order=integration_order,
    specific_body_force="0 -9.81",
    reference_temperature="T_ref",
)
prj.processes.set_constitutive_relation(type="MFront", behaviour="ModCamClay_semiExpl")
prj.processes.add_process_variable(
    process_variable="process_variable", process_variable_name="displacement"
)
state_names = [
    "EquivalentPlasticStrain",
    "PreConsolidationPressure",
    "VolumeRatio",
    "PlasticVolumetricStrain",
]
for name in ["sigma", "epsilon", *state_names]:
    prj.processes.add_secondary_variable(internal_name=name, output_name=name)

# Standard MCC properties and the two element fields saved in soil.vtu.
material = dict(
    PoissonRatio=xi,
    CriticalStateLineSlope=M,
    SwellingLineSlope=kappa,
    VirginConsolidationLineSlope=lam,
    CharacteristicPreConsolidationPressure=100.0,
)
for name, value in (
    material
    | dict(T_ref=293.15, dirichlet0=0, surface_traction=-surface_pressure * 1000)
).items():
    prj.parameters.add_parameter(name=name, type="Constant", value=value)
prj.parameters.add_parameter(name="displacement0", type="Constant", values="0 0")
for name in ["InitialVolumeRatio", "density"]:
    prj.parameters.set_mesh_parameter(
        name=name, type="MeshElement", field_name=name, mesh="soil"
    )
relation = "./processes/process/constitutive_relation"
prj.add_element(parent_xpath=relation, tag="material_properties")
for name in [*material, "InitialVolumeRatio"]:
    prj.add_element(
        parent_xpath=relation + "/material_properties",
        tag="material_property",
        attrib_list=["name", "parameter"],
        attrib_value_list=[name, name],
    )
prj.media.add_property(
    medium_id=0,
    phase_type="Solid",
    name="density",
    type="Parameter",
    parameter_name="density",
)

# Side rollers, fixed bottom and uniform surface pressure.
prj.process_variables.set_ic(
    process_variable_name="displacement",
    components=2,
    order=2,
    initial_condition="displacement0",
)
for boundary, component in [("left", 0), ("right", 0), ("bottom", 0), ("bottom", 1)]:
    prj.process_variables.add_bc(
        process_variable_name="displacement",
        type="Dirichlet",
        mesh=boundary,
        component=component,
        parameter="dirichlet0",
    )
prj.process_variables.add_bc(
    process_variable_name="displacement",
    type="Neumann",
    mesh="top",
    component=1,
    parameter="surface_traction",
)

# Hold the initial state under constant loading.
prj.time_loop.add_process(
    process="SD",
    nonlinear_solver_name="basic_newton",
    convergence_type="PerComponentDeltaX",
    norm_type="NORM2",
    abstols="1e-9 1e-9",
    time_discretization="BackwardEuler",
)
prj.time_loop.set_stepping(
    process="SD",
    type="IterationNumberBasedTimeStepping",
    t_initial=0,
    t_end=1,
    initial_dt=0.125,
    minimum_dt=1e-5,
    maximum_dt=0.125,
    number_iterations=[1, 8, 16, 30],
    multiplier=[1.2, 1.0, 0.75, 0.5],
)
outputs = [
    "displacement",
    "NodalForces",
    "sigma",
    "epsilon",
    *state_names,
    "sigma_ip",
    "epsilon_ip",
    "IntegrationPointMetaData",
]
outputs += [f"material_state_variable_{name}_ip" for name in state_names]
prj.time_loop.add_output(
    type="VTK",
    prefix="footing",
    suffix="_ts_{:timestep}_t_{:time}",
    data_mode="Binary",
    repeat=100000,
    each_steps=1000000,
    fixed_output_times=[0, 0.5, 1],
    variables=outputs,
)
prj.nonlinear_solvers.add_non_lin_solver(
    name="basic_newton",
    type="Newton",
    max_iter=60,
    linear_solver="general_linear_solver",
)
prj.linear_solvers.add_lin_solver(
    name="general_linear_solver", kind="eigen", solver_type="SparseLU", scaling="true"
)
prj.linear_solvers.add_lin_solver(
    name="general_linear_solver",
    kind="petsc",
    prefix="footing",
    parameters="-footing_ksp_type gmres -footing_pc_type lu -footing_ksp_rtol 1e-12",
)
# Placeholder for reference-field comparisons in the benchmark pipeline.
prj.add_block(blocktag="test_definition", parent_xpath=".")
prj.add_block(blocktag="vtkdiff", parent_xpath="./test_definition")
prj.write_input()


### 3.3. Initial integration-point fields

The analytical initial state is assigned at the OGS integration-point coordinates and stored in each case's `soil.vtu`. The required integration-point metadata are obtained through `ogstools.IPdata`, allowing stress, cap pressure, and specific volume to be supplied directly as initial fields rather than through lengthy expressions in the project file.

Density is constant within each layer and stored as an element field. The material property `InitialVolumeRatio` is evaluated at element centres and provides the reference volume used by the constitutive implementation [1]. It is distinct from the evolving `VolumeRatio` state, which is initialised directly at the integration points. The approximation associated with the element-centre evaluation is quantified below.

OGS reads the following initial integration-point fields from `soil.vtu`:

| Initial quantity | Field in `soil.vtu` |
|---|---|
| Stress tensor [Pa] | `sigma_ip` |
| Cap pressure [Pa] | `material_state_variable_PreConsolidationPressure_ip` |
| Specific volume [-] | `material_state_variable_VolumeRatio_ip` |

The time-zero read-back check below verifies that these prescribed fields are imported unchanged.

In [ ]:
# Ask OGS for its integration-point coordinates and ordering, without a simulation.
from tempfile import TemporaryDirectory

with TemporaryDirectory(dir=out_dir, prefix="ip_") as temporary:
    coordinates_file = Path(temporary) / "ip_coordinates.vtu"
    assert (
        cli.ipDataToPointCloud(
            "--log-level",
            "warn",
            O=integration_order,
            i=str(work / "soil.vtu"),
            o=str(coordinates_file),
        )
        == 0
    )
    ip_coordinates = pv.read(coordinates_file).points.copy()
cell_depth = -soil.cell_centers().points[:, 1]
layer_ids = np.clip(np.searchsorted(edges[1:], cell_depth), 0, len(layers) - 1)
volume_approximation = []


def make_project(ocr):
    folder = work / f"OCR_{ocr:g}"
    folder.mkdir(exist_ok=True)
    initial = soil.copy()
    sv, sh, pc, v = expected(-ip_coordinates[:, 1], ocr)
    # IPdata writes both the arrays and the metadata required by the OGS reader.
    fields = ot.mesh.IPdata(initial)
    fields.set(
        "sigma_ip",
        integration_order,
        4,
        np.column_stack([-sh, -sv, -sh, np.zeros_like(sv)]),
    )
    fields.set(
        "material_state_variable_PreConsolidationPressure_ip", integration_order, 1, pc
    )
    fields.set("material_state_variable_VolumeRatio_ip", integration_order, 1, v)
    initial.cell_data["density"] = gamma[layer_ids] / 9.81
    initial.cell_data["InitialVolumeRatio"] = expected(cell_depth, ocr)[3]
    # v0 is a material property, distinct from the exact evolving IP volume state.
    assert np.all(initial.cell_data["InitialVolumeRatio"] > 1)
    n_ip = len(v) // soil.n_cells
    assert n_ip * soil.n_cells == len(v), "This example requires a uniform T6 mesh."
    v0_ip = np.repeat(initial.cell_data["InitialVolumeRatio"], n_ip)
    volume_approximation.append(
        [ocr, abs(v0_ip - v).max(), (abs(v0_ip - v) / v).max() * 100]
    )
    initial.save(folder / "soil.vtu")

    model = ot.Project(
        input_file=out_dir / "ground_template.prj", output_file=folder / "ground.prj"
    )
    for i, name in enumerate(["soil", "left", "right", "bottom", "top"]):
        model.replace_text(
            "soil.vtu" if name == "soil" else f"../{name}.vtu",
            xpath="./meshes/mesh",
            occurrence=i,
        )
    model.write_input()
    return folder


### 3.4. Computation

Run all OCR cases and read the results with `ot.MeshSeries`. The time-zero check verifies the imported initial fields; the final-time check verifies completion of the hold.

In [ ]:
finals, points = {}, {}
for ocr in OCR_values:
    folder = make_project(ocr)
    out = folder / "output"
    out.mkdir(exist_ok=True)
    pvd = out / "footing.pvd"
    model = ot.Project(
        input_file=folder / "ground.prj", output_file=folder / "ground.prj"
    )
    model.run_model(
        logfile=out / "ogs.log", args=f'--log-level error -o "{out}"', write_logs=False
    )
    result = ot.MeshSeries(pvd)
    assert np.isclose(result.timevalues[-1], 1), "Hold did not finish."
    # Read-back check: imported IP fields must survive OGS initialization unchanged.
    assigned, start = pv.read(folder / "soil.vtu"), result.mesh(0)
    for field in ot.mesh.IPdata(assigned):
        assert np.allclose(
            start.field_data[field].ravel(),
            assigned.field_data[field].ravel(),
            rtol=0,
            atol=1e-9,
        ), f"IP initialization changed {field}"
    finals[ocr] = result.mesh(-1)
    points[ocr] = ot.mesh.to_ip_point_cloud(
        finals[ocr]
    )  # Actual material-point values.
print("Completed OCR cases:", OCR_values)

# Quantify the only field approximation; exact VolumeRatio_ip is checked below.
volume_table = pd.DataFrame(
    volume_approximation,
    columns=["OCR", "Max |v0_element − v0_exact|", "Max relative error [%]"],
)
display(volume_table)


## 4. Results

### 4.1. Stress profiles

The prescribed analytical profiles are compared with the OGS results extracted along the central vertical strip. Solid lines show the analytical solution and open markers show the numerical values; colour denotes OCR. The vertical effective stress is identical for all OCR cases because the geometry, unit weights, and surface surcharge are unchanged. In contrast, the horizontal effective stress and preconsolidation pressure vary with OCR because they depend on the assumed stress history.


In [ ]:
colours = dict(zip(OCR_values, plt.colormaps["tab10"].colors))
fig, axs = plt.subplots(1, 3, figsize=(11.5, 4.6), constrained_layout=True)
for ocr, ip in points.items():
    sample = ip.extract_points(abs(ip.points[:, 0] - width / 2) <= mesh_size / 2)
    z = -sample.points[:, 1]
    order = np.argsort(z)
    show = order[:: max(1, len(order) // 22)]
    sv, sh, pc, _ = expected(z, ocr)
    values = [
        -sample["sigma_ip"][:, 1],
        -sample["sigma_ip"][:, 0],
        sample["material_state_variable_PreConsolidationPressure_ip"].ravel(),
    ]
    for ax, numerical, target, title in zip(
        axs,
        values,
        [sv, sh, pc],
        ["Vertical stress", "Horizontal stress", "Cap pressure"],
    ):
        ax.plot(target[order] / 1000, z[order], color=colours[ocr])
        ax.plot(
            numerical[show] / 1000,
            z[show],
            "o",
            color=colours[ocr],
            markerfacecolor="none",
            markersize=4,
        )
        ax.set(title=title, xlabel="Pressure [kPa]", ylim=(depth, 0))
axs[0].set_ylabel("Depth [m]")
handles = [
    Line2D([], [], color=colours[ocr], label=f"OCR {ocr:g}") for ocr in OCR_values
]
handles += [
    Line2D([], [], color="black", label="Analytical reference"),
    Line2D(
        [],
        [],
        color="black",
        linestyle="none",
        marker="o",
        markerfacecolor="none",
        label="OGS",
    ),
]
fig.legend(
    handles=handles, loc="outside lower center", ncol=len(handles), frameon=False
)
plt.show()


### 4.2. Equilibrium and constitutive admissibility

A successful initialisation must satisfy three conditions: the prescribed stress field must be reproduced, the model must remain in global equilibrium, and the MCC state must remain admissible without unintended plastic evolution. The checks below evaluate these conditions across the full domain. Numerical tolerances are used only as acceptance criteria for the benchmark; they do not represent uncertainty in the material parameters.

In [ ]:
checks = []
for ocr, ip in points.items():
    mesh = finals[ocr]
    sv, sh, pc, v = expected(-ip.points[:, 1], ocr)
    target = np.column_stack([-sh, -sv, -sh, np.zeros_like(sv)])
    stress_error = abs(ip["sigma_ip"] - target).max()
    movement = abs(mesh["displacement"]).max()
    reaction = mesh["NodalForces"][np.isclose(mesh.points[:, 1], -depth), 1].sum()
    load = width * expected(depth, ocr)[0]
    cap_error = abs(
        ip["material_state_variable_PreConsolidationPressure_ip"].ravel() - pc
    ).max()
    volume_error = abs(ip["material_state_variable_VolumeRatio_ip"].ravel() - v).max()
    plastic = abs(ip["material_state_variable_EquivalentPlasticStrain_ip"]).max()
    passed = [
        stress_error < 0.1,
        movement < 1e-8 and abs(reaction - load) < 0.01,
        cap_error < 0.1 and volume_error < 1e-9 and plastic < 1e-9,
    ]
    checks.append(
        [
            ocr,
            *["PASS" if ok else "CHECK INPUTS" for ok in passed],
            movement,
            reaction,
            plastic,
        ]
    )
    assert all(passed), f"Review OCR {ocr} before proceeding."
check_table = pd.DataFrame(
    checks,
    columns=[
        "OCR",
        "Stresses match",
        "Ground stays still",
        "MCC unchanged",
        "Movement [m]",
        "Reaction [N/m]",
        "Plastic strain",
    ],
)
display(check_table.iloc[:, :4])


### 4.3. Local yield condition

For each OCR, the point with the highest mean effective compression is plotted against its MCC yield surface [1,2]:

$$
f=q^2+M^2p(p-p_c),\qquad \widehat f=f/p_c^2.
$$

Negative $\widehat f$ indicates a state inside the surface; zero indicates yield. OCR 1 lies close to the surface because of the small 1 Pa cap margin. Admissibility is also checked at every integration point, while unchanged material variables confirm that the hold introduces no plastic adjustment.

In [ ]:
# Evaluate full stress invariants at every IP; select maximum p separately for each OCR.
yield_rows = []
fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.8), constrained_layout=True)
for ax, ocr in zip(axs, OCR_values):
    ip = points[ocr]
    stress = ip["sigma_ip"]
    p_eff = -stress[:, :3].sum(axis=1) / 3
    dev = stress[:, :3] + p_eff[:, None]
    q_eff = np.sqrt(1.5 * ((dev**2).sum(axis=1) + 2 * stress[:, 3] ** 2))
    cap = ip["material_state_variable_PreConsolidationPressure_ip"].ravel()
    fhat = (q_eff**2 + M * M * p_eff * (p_eff - cap)) / cap**2
    assert (
        np.all(np.isfinite(fhat)) and fhat.max() <= 1e-10
    ), "State outside the MCC cap."
    idx = int(np.argmax(p_eff))  # Highest mean compression, not largest yield residual.
    x, y, _ = ip.points[idx]
    yield_rows.append(
        [
            ocr,
            idx,
            x,
            -y,
            p_eff[idx] / 1000,
            q_eff[idx] / 1000,
            cap[idx] / 1000,
            fhat[idx],
            fhat.max(),
        ]
    )
    p_curve = np.linspace(0, cap[idx], 200)
    ax.plot(
        p_curve / 1000,
        M * np.sqrt(p_curve * (cap[idx] - p_curve)) / 1000,
        color=colours[ocr],
        label="MCC yield surface",
    )
    ax.plot(
        p_eff[idx] / 1000,
        q_eff[idx] / 1000,
        "o",
        color="black",
        markersize=4,
        label="Selected material point",
    )
    ax.set(
        title=f"OCR {ocr:g}",
        xlabel="p [kPa]",
        ylabel="q [kPa]",
        xlim=(0, None),
        ylim=(0, None),
    )
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False)
plt.show()
yield_table = pd.DataFrame(
    yield_rows,
    columns=[
        "OCR",
        "IP index",
        "x [m]",
        "Depth [m]",
        "p [kPa]",
        "q [kPa]",
        "pc [kPa]",
        "f / pc²",
        "Maximum f / pc² in domain",
    ],
)
display(
    yield_table.style.format(
        {
            "x [m]": "{:.3f}",
            "Depth [m]": "{:.3f}",
            "p [kPa]": "{:.2f}",
            "q [kPa]": "{:.2f}",
            "pc [kPa]": "{:.2f}",
            "f / pc²": "{:.3e}",
            "Maximum f / pc² in domain": "{:.3e}",
        }
    )
)

# Compare displacement, force balance and yield admissibility.
fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.6), constrained_layout=True)
labels = check_table.OCR.astype(str)
axs[0].bar(labels, check_table["Movement [m]"], color=list(colours.values()))
axs[1].bar(labels, check_table["Reaction [N/m]"] / 1000, color=list(colours.values()))
axs[1].axhline(float(load) / 1000, color="black", ls="--", lw=1, label="Applied load")
axs[1].legend(frameon=False)
axs[2].plot(labels, yield_table["f / pc²"], "o", color="black", markersize=5)
axs[2].axhline(0, color="tab:red", ls="--", lw=1, label="Yield surface")
axs[2].legend(frameon=False, loc="lower left")
for ax, title, label in zip(
    axs,
    ["Ground motion", "Global equilibrium", "Local admissibility"],
    ["Maximum |u| [m]", "Bottom reaction [kN/m]", "f / pc² [-]"],
):
    ax.set(title=title, xlabel="OCR", ylabel=label)
plt.show()


## 5. Further use and development

This benchmark establishes a mechanically equilibrated and constitutively admissible Modified Cam-Clay state for horizontally layered ground. The resulting stress field, preconsolidation pressure, and specific volume provide a controlled starting point for subsequent geotechnical analyses. The same initialisation concept can support several classes of problems, but the continuation depends on whether the engineering operation loads, removes, or adds material.

### 5.1. Loading of the existing ground

If the soil domain remains unchanged, the equilibrated state can be used directly as the starting point for subsequent loading. Typical examples include:

- shallow or deep foundation loading,
- surface surcharge,
- embankment loading represented as an external load,
- changes in mechanical boundary conditions.

In these cases, the verified geostatic state is retained and the additional load is introduced on top of it.

### 5.2. Excavation and material removal

For excavation problems, the soil to be removed should first belong to the initial
ground and therefore receive the same consistent in-situ stress and MCC history as the
surrounding material.

The construction sequence then becomes

$$
\text{geostatic initialisation}
\rightarrow
\text{equilibrium hold}
\rightarrow
\text{material removal}
\rightarrow
\text{stress redistribution}.
$$

OGS already supports time-dependent removal of material through `<deactivated_subdomains>` [4,6]. The linear element-deactivation benchmark [4] demonstrates the basic mechanism, while the Kirsch example [5] illustrates its use for an excavation-type cavity problem. This provides a natural continuation of the present benchmark for problems such as:

- deep excavations,
- tunnel or cavity excavation,
- staged removal of soil,
- excavation followed by structural support activation.

Readers interested in this workflow can refer to the existing OGS element-deactivation
examples, including the linear mechanics deactivation benchmark and the Kirsch problem
with an excavated cavity.

### 5.3. Construction of new soil

For history-dependent constitutive models such as MCC, staged material addition requires more than activating previously inactive elements. Newly placed soil must enter the analysis with a compatible constitutive placement state and an appropriate placement reference before gravity and subsequent construction loading are applied.

A general staged material-activation workflow of this type is beyond the scope of the present benchmark and represents a natural direction for future OGS development.

Newly placed soil should not simply inherit the $K_0$/OCR state of the underlying natural ground. Instead, it requires its own placement state,

$$
\left\{
\boldsymbol{\sigma}'_{\mathrm{placement}},
p_{c,\mathrm{placement}},
v_{\mathrm{placement}}
\right\},
$$

followed by equilibration under self-weight.

For a constructed slope or embankment, the desired workflow is therefore

$$
\text{initialised foundation}
\rightarrow
\text{material placement}
\rightarrow
\text{gravity equilibration}.
$$

For staged construction, this can be repeated lift by lift,

$$
\text{foundation}
\rightarrow
L_1
\rightarrow
L_2
\rightarrow
\cdots
\rightarrow
L_n.
$$

Such a workflow requires not only activation of previously inactive elements, but also
a consistent constitutive state and placement reference at activation. This capability
is a natural extension of the current deactivation-based framework and is a subject of
ongoing development.

### 5.4. Potential applications

Once a reliable initial state and staged-construction workflow are available, the same
framework can be extended to:

- retaining-wall and anchored excavation analyses,
- embankments and earth dams,
- backfilling and reclamation,
- tunnel excavation and staged support installation,
- repository excavations and engineered barrier placement,
- construction-history studies with path-dependent constitutive models.

The present benchmark therefore represents the initial-state component of a broader
staged geomechanics workflow.

### 5.5. Scope and future development

The current notebook is limited to the geostatic initialisation of an existing, horizontally layered ground profile. It does not address:

- activation of newly placed soil,
- staged construction of fills,
- transfer of constitutive history between unrelated meshes,
- saturated or coupled hydro-mechanical initialisation,
- large-deformation geometry changes.

Future developments may link the verified initial state to excavation and material-placement workflows while preserving the complete constitutive history at integration points.

The guiding principle remains the same: every active material region should enter a subsequent engineering stage in a mechanically equilibrated and constitutively compatible state.


## References

[1] Silbermann, C., and Nagel, T. (2023). [Implementation of the Modified Cam Clay Model in MFront/OpenGeoSys](https://www.opengeosys.org/docs/benchmarks/small-deformations/modifiedcamclay/ModifiedCamClay_report.pdf). OpenGeoSys technical report describing the MCC formulation, MFront implementation, state variables, pressure-dependent elasticity, and numerical integration.

[2] Itasca. [Modified Cam-Clay Model](https://docs.itascacg.com/3dec700/common/models/camclay/doc/modelcamclay.html). Model documentation covering the MCC yield surface, hardening, preconsolidation pressure, specific volume, and initial-state relations.

[3] Bentley Systems / Seequent. [PLAXIS 2D Reference Manual, Version 2025.1](https://files.seequent.com/PLAXIS/Manuals/PLAXIS_2D/English/PLAXIS_2D_2_Reference%20Manual.pdf), Section 6.1.8.1, Eq. 6–42. Automatic initial lateral stress estimation using $K_0$, OCR, POP, and unloading/reloading elasticity.

[4] OpenGeoSys. [Linear; Element deactivation](https://www.opengeosys.org/6.5.9/docs/benchmarks/small-deformations/mechanics-linear-element_deactivation/). Benchmark demonstrating time-dependent element deactivation in the
`SmallDeformation` process.

[5] OpenGeoSys. [Solving Kirsch's problem using the release nodal force approach](https://www.opengeosys.org/stable/docs/benchmarks/small-deformations/kirsch/). Mechanics benchmark including an element-deactivation representation of cavity excavation.

[6] OpenGeoSys. [DeactivatedSubdomain documentation](https://doxygen.opengeosys.org/6.5.9/d0/d63/structProcessLib_1_1DeactivatedSubdomain.html). Documentation of space- and time-dependent subdomain deactivation.
